# 83 — SOC Analyst Copilot

## Goal
Build an agent that **triages security alerts**, **summarizes context**,
and **proposes likely next steps** for a Security Operations Center analyst.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Alert triage** | Classify severity and type |
| **Context gathering** | Enrich alerts with threat intel |
| **Response playbook** | Suggest investigation steps |
| **LangGraph** | Multi-node stateful agent |

## Stack
- `langgraph` — stateful graph
- `langchain-ollama` — LLM
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Simulated Alert Feed & Threat Intel

In [ ]:
# Simulated SIEM alerts
ALERTS = [
    {
        "id": "ALT-001",
        "timestamp": "2024-11-15T02:34:12Z",
        "source": "EDR",
        "title": "Suspicious PowerShell execution",
        "description": "PowerShell process spawned with encoded command flag (-enc) by user jsmith on workstation WS-FIN-042. Command attempts to download from external URL hxxps://evil-payload[.]xyz/stage2.ps1",
        "severity": "high",
        "host": "WS-FIN-042",
        "user": "jsmith",
        "department": "Finance"
    },
    {
        "id": "ALT-002",
        "timestamp": "2024-11-15T02:35:01Z",
        "source": "Firewall",
        "title": "Outbound connection to known C2 IP",
        "description": "WS-FIN-042 attempted outbound connection to 198.51.100.23:443, flagged as known Command & Control infrastructure (Cobalt Strike).",
        "severity": "critical",
        "host": "WS-FIN-042",
        "user": "jsmith",
        "department": "Finance"
    },
    {
        "id": "ALT-003",
        "timestamp": "2024-11-15T03:12:45Z",
        "source": "IAM",
        "title": "Multiple failed login attempts",
        "description": "15 failed login attempts for user admin_backup from IP 10.0.5.22 within 2 minutes. Account locked after threshold.",
        "severity": "medium",
        "host": "DC-PROD-01",
        "user": "admin_backup",
        "department": "IT"
    },
]

# Simulated threat intelligence database
THREAT_INTEL = {
    "198.51.100.23": {
        "type": "C2 Server",
        "malware_family": "Cobalt Strike",
        "first_seen": "2024-09-01",
        "confidence": "high",
        "associated_campaigns": ["FIN7", "APT29"]
    },
    "evil-payload.xyz": {
        "type": "Malware Distribution",
        "malware_family": "Unknown Loader",
        "first_seen": "2024-11-10",
        "confidence": "medium",
        "associated_campaigns": ["Unknown"]
    }
}

# Simulated asset inventory
ASSET_DB = {
    "WS-FIN-042": {"type": "Workstation", "os": "Windows 11", "criticality": "high", "last_patch": "2024-11-01"},
    "DC-PROD-01": {"type": "Domain Controller", "os": "Windows Server 2022", "criticality": "critical", "last_patch": "2024-10-15"},
}

print(f"Loaded {len(ALERTS)} alerts, {len(THREAT_INTEL)} threat intel entries, {len(ASSET_DB)} assets")

## Step 2 — LangGraph SOC Workflow

```
ingest_alerts → enrich_context → triage → recommend_actions → END
```

In [ ]:
class SOCState(TypedDict):
    raw_alerts: list
    enriched_alerts: list
    triage_results: list
    recommendations: list
    incident_summary: str

# Node 1: Ingest and correlate alerts
def ingest_alerts(state: SOCState) -> SOCState:
    print("📥 Ingesting alerts...")
    alerts = state["raw_alerts"]
    
    # Group by host to find correlations
    host_groups = {}
    for a in alerts:
        host = a.get("host", "unknown")
        host_groups.setdefault(host, []).append(a)
    
    for host, group in host_groups.items():
        print(f"  Host {host}: {len(group)} alert(s)")
        if len(group) > 1:
            print(f"    ⚠️ CORRELATED — multiple alerts from same host!")
    
    return state

# Node 2: Enrich with threat intel and asset data
def enrich_context(state: SOCState) -> SOCState:
    print("\n🔍 Enriching with threat intel & asset data...")
    enriched = []
    
    for alert in state["raw_alerts"]:
        enrichment = {"alert": alert, "threat_intel": [], "asset_info": None}
        
        # Check threat intel for IPs and domains in description
        for ioc, intel in THREAT_INTEL.items():
            if ioc.replace(".", "[\.]").replace(".", ".") in alert["description"] or ioc.replace(".", "[.]") in alert["description"] or ioc in alert["description"]:
                enrichment["threat_intel"].append({"ioc": ioc, **intel})
                print(f"  🎯 Alert {alert['id']}: IOC match → {ioc} ({intel['type']})")
        
        # Check asset database
        host = alert.get("host", "")
        if host in ASSET_DB:
            enrichment["asset_info"] = ASSET_DB[host]
            print(f"  🖥️ Alert {alert['id']}: Asset → {host} (criticality: {ASSET_DB[host]['criticality']})")
        
        enriched.append(enrichment)
    
    return {**state, "enriched_alerts": enriched}

print("Nodes 1-2 defined")

In [ ]:
# Node 3: LLM-powered triage
def triage(state: SOCState) -> SOCState:
    print("\n🚦 Triaging alerts...")
    triage_results = []
    
    for enriched in state["enriched_alerts"]:
        alert = enriched["alert"]
        context = json.dumps(enriched, indent=2, default=str)
        
        msg = llm.invoke([
            SystemMessage(content="""You are a senior SOC analyst. Triage this security alert.
Assess:
1. Is this a true positive, false positive, or needs investigation?
2. MITRE ATT&CK technique (if applicable)
3. Urgency: P1 (immediate), P2 (4 hours), P3 (24 hours), P4 (informational)
4. One-line summary

Return JSON with keys: verdict, mitre_technique, priority, summary
Return ONLY JSON, no other text."""),
            HumanMessage(content=f"Alert + context:\n{context}")
        ])
        
        raw = msg.content
        if "<think>" in raw:
            raw = raw.split("</think>")[-1].strip()
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
            raw = raw.strip()
        
        try:
            start = raw.find("{")
            end = raw.rfind("}") + 1
            result = json.loads(raw[start:end])
        except (json.JSONDecodeError, ValueError):
            result = {"verdict": "needs_investigation", "mitre_technique": "unknown", "priority": "P2", "summary": alert["title"]}
        
        result["alert_id"] = alert["id"]
        triage_results.append(result)
        
        priority = result.get("priority", "?")
        print(f"  [{priority}] {alert['id']}: {result.get('verdict', '?')} — {result.get('summary', alert['title'])[:60]}")
    
    return {**state, "triage_results": triage_results}

print("Node 3 (triage) defined")

In [ ]:
# Node 4: Recommend next steps
def recommend_actions(state: SOCState) -> SOCState:
    print("\n📋 Generating recommendations...")
    
    all_context = json.dumps({
        "enriched_alerts": state["enriched_alerts"],
        "triage_results": state["triage_results"]
    }, indent=2, default=str)
    
    msg = llm.invoke([
        SystemMessage(content="""You are a senior SOC analyst writing an incident summary.
Based on the triaged alerts, provide:
1. Overall incident assessment (1-2 sentences)
2. Recommended immediate actions (numbered list)
3. Escalation recommendation (yes/no and to whom)
4. Evidence to preserve

Be concise and actionable. Do not include any thinking."""),
        HumanMessage(content=f"Triaged alerts:\n{all_context}")
    ])
    
    summary = msg.content
    if "<think>" in summary:
        summary = summary.split("</think>")[-1].strip()
    
    return {**state, "incident_summary": summary}

print("Node 4 (recommend_actions) defined")

In [ ]:
# Build the graph
graph = StateGraph(SOCState)

graph.add_node("ingest", ingest_alerts)
graph.add_node("enrich", enrich_context)
graph.add_node("triage", triage)
graph.add_node("recommend", recommend_actions)

graph.set_entry_point("ingest")
graph.add_edge("ingest", "enrich")
graph.add_edge("enrich", "triage")
graph.add_edge("triage", "recommend")
graph.add_edge("recommend", END)

soc_app = graph.compile()
print("SOC Copilot graph compiled!")

In [ ]:
# Run the SOC copilot
result = soc_app.invoke({
    "raw_alerts": ALERTS,
    "enriched_alerts": [],
    "triage_results": [],
    "recommendations": [],
    "incident_summary": ""
})

print("\n" + "=" * 70)
print("INCIDENT RESPONSE SUMMARY")
print("=" * 70)
print(result["incident_summary"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Alert correlation** | Group by host to spot patterns |
| **Threat intel enrichment** | Match IOCs against known threats |
| **LLM triage** | Classify verdict, priority, MITRE ATT&CK |
| **Playbook generation** | Actionable response recommendations |

## Architecture
```
SIEM Alerts → Ingest & Correlate → Enrich (Threat Intel + Assets)
    → Triage (LLM: verdict + priority) → Recommend (LLM: next steps) → Report
```

## Extending This Project
- Integrate real SIEM feeds (Splunk, Sentinel)
- Add automated containment actions (isolate host, block IP)
- Track analyst feedback to improve triage accuracy
- Add SOAR (Security Orchestration) integration